# 05. Graph Model Comparison

This notebook trains and compares all graph baselines under the same split and feature setup:

- `GCN`
- `GraphSAGE`
- `GAT`
- `GGNN`

Each model uses its own default hyperparameter preset, but all models share the same dataset split, evaluation protocol, threshold tuning, and reporting metrics.
The default comparison is stricter by keeping `ADD_GRAPH_STATS = False`.


In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd

module_dir_candidates = [Path.cwd(), Path.cwd() / "graph_models", Path.cwd().parent / "graph_models"]
module_dir = next(path for path in module_dir_candidates if (path / "graph_model_utils.py").exists())
sys.path.append(str(module_dir))

from graph_model_utils import (
    build_graph_dataset,
    find_best_threshold,
    get_model,
    plot_comparison_bars,
    plot_evaluation_dashboard,
    predict_probabilities,
    set_seed,
    train_model,
    evaluate_probabilities,
)


In [ ]:
FEATURE_GROUP = "features"
ADD_GRAPH_STATS = False
RANDOM_STATE = 42
THRESHOLD_OBJECTIVE = "f1"

MODEL_CONFIGS = {
    "gcn": {"hidden_dim": 64, "dropout": 0.2, "learning_rate": 1e-3, "epochs": 100, "patience": 15},
    "graphsage": {"hidden_dim": 64, "dropout": 0.2, "learning_rate": 1e-3, "epochs": 100, "patience": 15},
    "gat": {"hidden_dim": 32, "dropout": 0.3, "learning_rate": 7.5e-4, "epochs": 120, "patience": 18},
    "ggnn": {"hidden_dim": 64, "dropout": 0.2, "learning_rate": 1e-3, "epochs": 100, "patience": 15},
}

set_seed(RANDOM_STATE)


In [ ]:
data = build_graph_dataset(
    feature_group=FEATURE_GROUP,
    add_graph_stats=ADD_GRAPH_STATS,
    random_state=RANDOM_STATE,
)

display(pd.DataFrame([data["graph_summary"]]))
display(data["split_df"])
print("Device:", data["device"])
print("Number of input features:", len(data["feature_cols"]))


In [ ]:
results = []
fitted = {}
histories = {}

for model_name, cfg in MODEL_CONFIGS.items():
    set_seed(RANDOM_STATE)
    model = get_model(
        model_name=model_name,
        in_dim=data["features"].shape[1],
        hidden_dim=cfg["hidden_dim"],
        dropout=cfg["dropout"],
    )

    start = time.perf_counter()
    best_model, history_df = train_model(
        model=model,
        data=data,
        learning_rate=cfg["learning_rate"],
        weight_decay=1e-4,
        epochs=cfg["epochs"],
        patience=cfg["patience"],
        threshold_objective=THRESHOLD_OBJECTIVE,
    )
    elapsed = time.perf_counter() - start

    val_true, val_prob = predict_probabilities(best_model, data, "val_mask")
    threshold = find_best_threshold(val_true, val_prob, objective=THRESHOLD_OBJECTIVE)
    test_true, test_prob = predict_probabilities(best_model, data, "test_mask")
    metrics = evaluate_probabilities(test_true, test_prob, threshold)

    results.append(
        {
            "model": model_name,
            "epochs_ran": int(history_df["epoch"].max()),
            "train_seconds": elapsed,
            **metrics,
        }
    )
    fitted[model_name] = {
        "model": best_model,
        "threshold": threshold,
        "test_true": test_true,
        "test_prob": test_prob,
        "history": history_df,
    }
    histories[model_name] = history_df


In [ ]:
results_df = pd.DataFrame(results).sort_values(["PR-AUC", "F1"], ascending=False)
display(results_df)


In [ ]:
plot_comparison_bars(results_df)


In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Best graph model:", best_model_name)

best_result = fitted[best_model_name]
plot_evaluation_dashboard(
    y_true=best_result["test_true"],
    y_prob=best_result["test_prob"],
    threshold=best_result["threshold"],
    title_prefix=best_model_name.upper(),
)


In [ ]:
summary_cols = [
    "model",
    "PR-AUC",
    "ROC-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Balanced-Accuracy",
    "MCC",
    "Brier-Score",
    "Accuracy",
    "Precision@K",
    "Recall@K",
    "epochs_ran",
    "train_seconds",
]
results_df[summary_cols]


## Notes

This notebook is the main leaderboard for graph models.
It defaults to a stricter feature-only setup by keeping `ADD_GRAPH_STATS = False`.
If you later tune hyperparameters, keep the same split and threshold-selection protocol so the comparison remains valid.
